# as-strided-noncontig-source — worked example 1: Slicing a tensor changes shape but not stride

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `as-strided-noncontig-source`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Concept

**Stride** is the number of storage elements to skip to advance one step along an axis. A *basic slice* like `x[:, 2:6]` produces a **view**: it changes the shape and the storage *offset*, but it keeps the same per-axis strides as the parent because the underlying row-major layout is unchanged. The slice is only contiguous if the columns it keeps happen to be a full row width.

## Worked solution

We want to show that narrowing the columns of a contiguous `(H, W)` tensor leaves the strides untouched.

1. Build a contiguous `(4, 8)` tensor with `t.arange(32).reshape(4, 8)`. Its stride is `(8, 1)` — one row down skips 8 elements, one column over skips 1.
2. Take `sl = x[:, 2:6]`. This keeps all 4 rows but only columns 2..5, giving shape `(4, 4)`.
3. Crucially, `sl.stride()` is still `(8, 1)`: to move down a row in `sl` you still skip a full *original* row of 8 elements, because the data was never copied — only the starting offset moved to column 2.
4. Because the per-row jump (8) no longer equals the new row width (4), `sl` is **not contiguous**. We confirm with `is_contiguous()`.
5. We return the stride tuple and the contiguity flag so the printed result makes the invariant visible: shape shrank, stride did not.

In [ ]:
def stride_after_narrow(x: Tensor) -> tuple:
    sl = x[:, 2:6]
    return (tuple(sl.stride()), sl.is_contiguous())

x = t.arange(32, dtype=t.float32).reshape(4, 8)
stride, contig = stride_after_narrow(x)
print('slice shape :', tuple(x[:, 2:6].shape))
print('slice stride:', stride)
print('contiguous? :', contig)